# No probado " cuando se tienen los videos en la carpeta Assets/videos"

In [ ]:
from pathlib import Path
import re
import cv2
import json

# ==========================================================
# CONFIGURACIÓN DEL PROYECTO
# ==========================================================

# Estructura detectada:
# PROYECTOFINAL/
# ├── Assets/
# │   ├── frames_ball/
# │   └── videos/
# ├── extract_frames.ipynb
# └── ...

# Carpeta donde están los videos
VIDEOS_ROOT = Path("Assets/videos")

# Carpeta donde se guardarán los frames
OUTPUT_BASE = Path("Assets/frames_ball")

# ==========================================================
# PARÁMETROS
# ==========================================================

# Máximo de partidos por cancha
MAX_PARTIDOS = 7

# Guardar 1 frame cada N frames
FRAME_STEP = 5

# None = guardar todos
MAX_FRAMES_A_GUARDAR = None

# Calidad JPG
JPG_QUALITY = 95

# Sobrescribir frames existentes
SOBRESCRIBIR = False

# Extensiones permitidas
VIDEO_EXTENSIONS = {".mp4", ".mov", ".avi", ".mkv"}

# ==========================================================
# UTILIDADES
# ==========================================================

def obtener_canchas(root: Path):
    """
    Busca carpetas tipo:
    Cancha_1
    Cancha_2
    """
    patron = re.compile(r"Cancha_(\d+)$", re.IGNORECASE)

    canchas = []
    for d in root.iterdir():
        if d.is_dir():
            m = patron.match(d.name)
            if m:
                canchas.append((int(m.group(1)), d))

    canchas.sort(key=lambda x: x[0])

    return [d for _, d in canchas]


def obtener_partidos(cancha_dir: Path, max_partidos: int):
    """
    Busca carpetas:
    Partido_1
    Partido_2
    """
    patron = re.compile(r"Partido_(\d+)$", re.IGNORECASE)

    partidos = []

    for d in cancha_dir.iterdir():
        if d.is_dir():
            m = patron.match(d.name)
            if m:
                partidos.append((int(m.group(1)), d))

    partidos.sort(key=lambda x: x[0])

    return partidos[:max_partidos]


def obtener_puntos(partido_dir: Path):
    """
    Busca videos:
    Punto_1.mp4
    Punto_2.mp4
    """

    patron = re.compile(r"Punto_(\d+)", re.IGNORECASE)

    puntos = []

    for archivo in partido_dir.iterdir():
        if archivo.is_file() and archivo.suffix.lower() in VIDEO_EXTENSIONS:

            m = patron.match(archivo.stem)

            if m:
                puntos.append((int(m.group(1)), archivo))

    puntos.sort(key=lambda x: x[0])

    return puntos


# ==========================================================
# EXTRACCIÓN
# ==========================================================

def extraer_frames(
    cancha_name: str,
    partido_num: int,
    punto_num: int,
    video_path: Path,
    frame_step: int = 5,
    max_frames_a_guardar=None,
    jpg_quality: int = 95,
    sobrescribir: bool = False,
):

    output_dir = (
        OUTPUT_BASE
        / cancha_name
        / f"Partido_{partido_num}"
        / f"Punto_{punto_num}"
    )

    metadata_path = output_dir / "metadata.json"

    if not video_path.exists():
        print(f"      ⚠️ Video no encontrado: {video_path}")
        return output_dir, 0

    output_dir.mkdir(parents=True, exist_ok=True)

    frames_existentes = list(output_dir.glob("*.jpg"))

    # Evitar reprocesar
    if frames_existentes and not sobrescribir:
        print(f"      ⏭️ Ya existen {len(frames_existentes)} frames")
        return output_dir, 0

    # Borrar si sobrescribe
    if sobrescribir:
        for f in frames_existentes:
            f.unlink()

    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        print(f"      ❌ No se pudo abrir: {video_path}")
        return output_dir, 0

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    duration = total_frames / fps if fps else 0

    print(
        f"      📹 {video_path.name} | "
        f"{total_frames} frames | "
        f"{fps:.1f} fps | "
        f"{duration:.1f}s"
    )

    frame_idx = 0
    saved_idx = 0

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        if frame_idx % frame_step == 0:

            if (
                max_frames_a_guardar is not None
                and saved_idx >= max_frames_a_guardar
            ):
                break

            filename = (
                output_dir
                / f"{cancha_name}_P{partido_num}_PT{punto_num}_F{frame_idx:06d}.jpg"
            )

            cv2.imwrite(
                str(filename),
                frame,
                [cv2.IMWRITE_JPEG_QUALITY, jpg_quality]
            )

            saved_idx += 1

        frame_idx += 1

    cap.release()

    metadata = {
        "cancha": cancha_name,
        "partido_num": partido_num,
        "punto_num": punto_num,
        "video_path": str(video_path),
        "output_dir": str(output_dir),
        "total_frames_video": total_frames,
        "fps": fps,
        "duration_seconds": duration,
        "width": width,
        "height": height,
        "frame_step": frame_step,
        "frames_guardados": saved_idx,
        "jpg_quality": jpg_quality,
    }

    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=4, ensure_ascii=False)

    print(f"      ✅ {saved_idx} frames guardados")

    return output_dir, saved_idx


# ==========================================================
# MAIN
# ==========================================================

OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

if not VIDEOS_ROOT.exists():
    raise FileNotFoundError(
        f"No existe la carpeta de videos: {VIDEOS_ROOT}"
    )

print("=" * 70)
print("🎾 EXTRACCIÓN DE FRAMES")
print("=" * 70)

canchas = obtener_canchas(VIDEOS_ROOT)

if not canchas:
    print("⚠️ No se encontraron carpetas Cancha_X")

total_frames_global = 0
resumen_global = []

for cancha_dir in canchas:

    cancha_name = cancha_dir.name

    print(f"\n🏟️ {cancha_name}")

    partidos = obtener_partidos(
        cancha_dir,
        MAX_PARTIDOS
    )

    if not partidos:
        print("   ⚠️ Sin partidos")
        continue

    print(
        f"   🎾 Partidos encontrados: "
        f"{[p[0] for p in partidos]}"
    )

    for partido_num, partido_dir in partidos:

        print(f"\n   📂 Partido_{partido_num}")

        puntos = obtener_puntos(partido_dir)

        if not puntos:
            print("      ⚠️ Sin puntos")
            continue

        for punto_num, video_path in puntos:

            print(f"      📌 Punto_{punto_num}")

            _, frames_guardados = extraer_frames(
                cancha_name=cancha_name,
                partido_num=partido_num,
                punto_num=punto_num,
                video_path=video_path,
                frame_step=FRAME_STEP,
                max_frames_a_guardar=MAX_FRAMES_A_GUARDAR,
                jpg_quality=JPG_QUALITY,
                sobrescribir=SOBRESCRIBIR,
            )

            total_frames_global += frames_guardados

            resumen_global.append({
                "cancha": cancha_name,
                "partido": partido_num,
                "punto": punto_num,
                "frames": frames_guardados,
                "video": str(video_path),
            })

# ==========================================================
# RESUMEN
# ==========================================================

print("\n" + "=" * 70)
print("✅ PROCESO COMPLETADO")
print("=" * 70)

print(f"🎾 Puntos procesados : {len(resumen_global)}")
print(f"🖼️ Frames nuevos     : {total_frames_global}")

print("\n🚀 Carpetas listas:")

carpetas_unicas = sorted({
    str(
        OUTPUT_BASE
        / e["cancha"]
        / f"Partido_{e['partido']}"
        / f"Punto_{e['punto']}"
    )
    for e in resumen_global
    if e["frames"] > 0
})

for c in carpetas_unicas:
    print(f"   {c}")

print("=" * 70)

# En caso de tener usb " Modificar de acuerdo al OS" " Caso linux

In [ ]:
from pathlib import Path
import re
import cv2
import json
import os

# ==========================================================
# CONFIGURACIÓN
# ==========================================================

# Raíz donde están las carpetas principales (ajusta si es necesario)
BASE_ROOTS = [
    Path("/media/lemarsito/F0D66CB9D66C81A8/Processed"),
    Path("/media/lemarsito/F0D66CB9D66C81A8/Videos_Processed"),
]

# Cuántos partidos procesar por cancha
MAX_PARTIDOS = 7

# Cada cuántos frames guardar una imagen
FRAME_STEP = 5

# Pon None para extraer todos los posibles
MAX_FRAMES_A_GUARDAR = None

# Calidad del JPG
JPG_QUALITY = 95

# Si True, borra los frames anteriores del mismo Punto antes de extraer
SOBRESCRIBIR = False

# Carpeta de salida base
OUTPUT_BASE = Path("data/frames_ball")


# ==========================================================
# UTILIDADES
# ==========================================================

def obtener_canchas(root: Path):
    """Devuelve subcarpetas tipo Cancha_X dentro de root."""
    patron = re.compile(r"Cancha_\d+$", re.IGNORECASE)
    canchas = sorted(
        [d for d in root.iterdir() if d.is_dir() and patron.match(d.name)],
        key=lambda d: int(re.search(r"\d+", d.name).group())
    )
    return canchas


def obtener_partidos(cancha_dir: Path, max_partidos: int):
    """Devuelve hasta max_partidos subcarpetas tipo Partido_X."""
    patron = re.compile(r"Partido_(\d+)$", re.IGNORECASE)
    partidos = []
    for d in cancha_dir.iterdir():
        if d.is_dir():
            m = patron.match(d.name)
            if m:
                partidos.append((int(m.group(1)), d))
    partidos.sort(key=lambda x: x[0])
    return partidos[:max_partidos]


def obtener_puntos(partido_dir: Path):
    """Devuelve todos los videos Punto_X.mp4 dentro de un partido."""
    patron = re.compile(r"Punto_(\d+)\.mp4$", re.IGNORECASE)
    puntos = []
    for archivo in partido_dir.iterdir():
        if archivo.is_file():
            m = patron.match(archivo.name)
            if m:
                puntos.append((int(m.group(1)), archivo))
    puntos.sort(key=lambda x: x[0])
    return puntos


# ==========================================================
# EXTRACCIÓN DE FRAMES
# ==========================================================

def extraer_frames(
    cancha_name: str,
    partido_num: int,
    punto_num: int,
    video_path: Path,
    frame_step: int = 5,
    max_frames_a_guardar=None,
    jpg_quality: int = 95,
    sobrescribir: bool = False,
):
    output_dir = OUTPUT_BASE / cancha_name / f"Partido_{partido_num}" / f"Punto_{punto_num}"
    metadata_path = output_dir / "metadata.json"

    if not video_path.exists():
        print(f"  ⚠️  Video no encontrado: {video_path}")
        return output_dir, 0

    output_dir.mkdir(parents=True, exist_ok=True)

    frames_existentes = list(output_dir.glob("*.jpg"))

    if frames_existentes and not sobrescribir:
        print(f"  ⏭️  Ya existen {len(frames_existentes)} frames — omitiendo (SOBRESCRIBIR=False)")
        return output_dir, 0

    if sobrescribir:
        for f in frames_existentes:
            f.unlink()

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"  ❌ No se pudo abrir: {video_path}")
        return output_dir, 0

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps         = cap.get(cv2.CAP_PROP_FPS)
    width       = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration    = total_frames / fps if fps else 0

    print(f"  📹 {video_path.name}  |  {total_frames} frames  |  {fps:.1f} fps  |  {duration:.1f}s  |  {width}x{height}")

    frame_idx = 0
    saved_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % frame_step == 0:
            if max_frames_a_guardar is not None and saved_idx >= max_frames_a_guardar:
                break

            filename = output_dir / f"{cancha_name}_Partido_{partido_num}_Punto_{punto_num}_frame_{frame_idx:06d}.jpg"
            cv2.imwrite(str(filename), frame, [cv2.IMWRITE_JPEG_QUALITY, jpg_quality])
            saved_idx += 1

        frame_idx += 1

    cap.release()

    metadata = {
        "cancha":               cancha_name,
        "partido_num":          partido_num,
        "punto_num":            punto_num,
        "video_path":           str(video_path),
        "output_dir":           str(output_dir),
        "total_frames_video":   total_frames,
        "fps":                  fps,
        "duration_seconds":     duration,
        "width":                width,
        "height":               height,
        "frame_step":           frame_step,
        "frames_guardados":     saved_idx,
        "jpg_quality":          jpg_quality,
    }
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=4, ensure_ascii=False)

    print(f"  ✅ {saved_idx} frames guardados → {output_dir}")
    return output_dir, saved_idx


# ==========================================================
# EJECUCIÓN PRINCIPAL
# ==========================================================

OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

resumen_global = []
total_frames_global = 0

for root in BASE_ROOTS:
    if not root.exists():
        print(f"\n⚠️  Carpeta raíz no encontrada, se omite: {root}")
        continue

    print("\n" + "=" * 70)
    print(f"📂 RAÍZ: {root}")
    print("=" * 70)

    canchas = obtener_canchas(root)
    if not canchas:
        print(f"  ⚠️  No se encontraron carpetas Cancha_X en {root}")
        continue

    for cancha_dir in canchas:
        cancha_name = cancha_dir.name
        print(f"\n🏟️  {cancha_name}")

        partidos = obtener_partidos(cancha_dir, MAX_PARTIDOS)
        if not partidos:
            print(f"  ⚠️  Sin partidos en {cancha_dir}")
            continue

        print(f"  Partidos encontrados: {[p[0] for p in partidos]}")

        for partido_num, partido_dir in partidos:
            print(f"\n  🎾 Partido {partido_num}")

            puntos = obtener_puntos(partido_dir)
            if not puntos:
                print(f"    ⚠️  Sin puntos en {partido_dir}")
                continue

            for punto_num, video_path in puntos:
                print(f"    📌 Punto {punto_num}")
                _, frames_guardados = extraer_frames(
                    cancha_name=cancha_name,
                    partido_num=partido_num,
                    punto_num=punto_num,
                    video_path=video_path,
                    frame_step=FRAME_STEP,
                    max_frames_a_guardar=MAX_FRAMES_A_GUARDAR,
                    jpg_quality=JPG_QUALITY,
                    sobrescribir=SOBRESCRIBIR,
                )
                total_frames_global += frames_guardados
                resumen_global.append({
                    "root":        str(root),
                    "cancha":      cancha_name,
                    "partido":     partido_num,
                    "punto":       punto_num,
                    "frames":      frames_guardados,
                    "video":       str(video_path),
                })

# ==========================================================
# RESUMEN FINAL
# ==========================================================

print("\n" + "=" * 70)
print("✅  PROCESO COMPLETADO")
print("=" * 70)
print(f"📦 Raíces procesadas : {[str(r) for r in BASE_ROOTS]}")
print(f"🎾 Puntos procesados : {len(resumen_global)}")
print(f"🖼️  Frames nuevos     : {total_frames_global}")
print("=" * 70)

print("\n🚀 Carpetas listas para Roboflow:")
carpetas_unicas = sorted({
    str(OUTPUT_BASE / e["cancha"] / f"Partido_{e['partido']}" / f"Punto_{e['punto']}")
    for e in resumen_global if e["frames"] > 0
})
for c in carpetas_unicas:
    print(f"  {c}")